# FacemoodClassifier — Depression-related facial expression classification

**Team:** Sara Bedulla, Kejdi Shahu

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaraBedulla/FacemoodClassifier/blob/main/notebooks/FacemoodClassifier_colab.ipynb)

This notebook runs the full pipeline from the [project repository](https://github.com/SaraBedulla/FacemoodClassifier):
data preparation → exploratory analysis → training → results.

It works in two places with the same code:
- **Google Colab**: clones the repo and downloads FER-2013 automatically. Use *Runtime → Change runtime type → T4 GPU*.
- **Locally** (PyCharm / Jupyter): uses the repo you already have and the dataset in `data/raw/`.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("FacemoodClassifier").exists():
        !git clone https://github.com/SaraBedulla/FacemoodClassifier.git
    %cd FacemoodClassifier
    # Colab already has TensorFlow, scikit-learn, pandas, matplotlib and pillow.
    !pip install -q kagglehub
else:
    # Running locally from notebooks/: move to the project root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")

print("Project root:", Path.cwd())

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow)")

## 2. Dataset

**FER-2013** ([Kaggle](https://www.kaggle.com/datasets/msambare/fer2013)): 35,887 grayscale 48×48 face images,
7 emotions (angry, disgust, fear, happy, sad, surprise, neutral). A stratified 10% of the training set is held out
for validation, giving 25,838 train / 2,871 validation / 7,178 test images.

In [ ]:
local_source = Path("data/raw/fer2013")
if local_source.exists():
    source = local_source
else:
    import kagglehub
    source = Path(kagglehub.dataset_download("msambare/fer2013"))
print("Dataset source:", source)

In [ ]:
!python -m src.data.prepare --dataset fer2013 --source "{source}"

## 3. Exploratory data analysis

In [ ]:
!python -m src.data.eda --dataset fer2013

In [ ]:
from IPython.display import Image, display

eda_dir = Path("results/figures/eda/fer2013")
for name in ["class_distribution_emotion7", "class_distribution_affect3", "samples", "mean_faces"]:
    print(name)
    display(Image(filename=str(eda_dir / f"{name}.png"), width=700))

## 4. Training — CNN from scratch

Two label schemes: `emotion7` (all 7 emotions) and `affect3` (negative / neutral / positive).
Each run keeps the weights with the lowest validation loss (early stopping, patience 10).

In [ ]:
!python -m src.training.train --model scratch_cnn --labels emotion7

In [ ]:
!python -m src.training.train --model scratch_cnn --labels affect3

## 5. Training and validation accuracy / loss

In [ ]:
import json
import pandas as pd

runs = sorted(Path("results/runs").glob("fer2013_*"))
summaries = [json.loads((r / "train_summary.json").read_text()) for r in runs if (r / "train_summary.json").exists()]
display(pd.DataFrame(summaries))

for r in runs:
    curves = r / "training_curves.png"
    if curves.exists():
        print(r.name)
        display(Image(filename=str(curves), width=800))

## 6. Test-set evaluation

Confusion matrix and sample predictions (correct and incorrect) — added in the next step.

## 7. Save results

In [ ]:
# Download all figures, tables and run summaries as one zip (Colab only).
if IN_COLAB:
    !zip -qr results.zip results -x "*.keras"
    from google.colab import files
    files.download("results.zip")